<a href="https://colab.research.google.com/github/JordanRD/PT01_GenerativeAI/blob/main/Project_Team_Group_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install transformers torch datasets accelerate

import torch
from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

model_name = "gpt2"

print("Memuat Tokenizer dan Model bawaan...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Mengunduh dataset cerita anak (TinyStories)...")
dataset = load_dataset("roneneldan/TinyStories", split="train[:1%]")


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Memproses tokenisasi pada dataset...")

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-storybook",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("\nMemulai proses Fine-Tuning...")
trainer.train()
print("Fine-Tuning Selesai!\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_story(age, genre, theme, strategy="top_p"):
    prompt = f"Write a {genre} story for a {age} year old about {theme}."
    print(f"[Prompt Input]: {prompt}")

    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    gen_kwargs = {
        "input_ids": input_ids,
        "max_length": 150,
        "no_repeat_ngram_size": 2,
        "pad_token_id": tokenizer.eos_token_id,
        "do_sample": True,
    }

    if strategy == "top_k":
        gen_kwargs["top_k"] = 40
        gen_kwargs["temperature"] = 0.7
        print(f"--- Hasil Generasi (Top-K Sampling) ---")

    elif strategy == "top_p":
        gen_kwargs["top_p"] = 0.9
        gen_kwargs["temperature"] = 0.8
        print(f"--- Hasil Generasi (Top-p / Nucleus Sampling) ---")

    output_ids = model.generate(**gen_kwargs)

    story = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return story

umur = 7
genre_cerita = "fantasy"
tema_cerita = "a little dragon who wants to be a hero"

cerita_top_k = generate_story(umur, genre_cerita, tema_cerita, strategy="top_k")
print(cerita_top_k)

print("\n" + "="*60 + "\n")

cerita_top_p = generate_story(umur, genre_cerita, tema_cerita, strategy="top_p")
print(cerita_top_p)

d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Memuat Tokenizer dan Model bawaan...


d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jorda\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning

Mengunduh dataset cerita anak (TinyStories)...


d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jorda\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 435236.68 examples/s]


...

Map: 100%|██████████| 21197/21197 [00:03<00:00, 6510.76 examples/s]



Memulai proses Fine-Tuning...


d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.199048
200,2.037072
300,1.979885
400,1.946627
500,1.939094
600,1.909492


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


In [ ]:
try:
    test_args = TrainingArguments(
        output_dir="./gpt2-storybook",
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
    )
    print("TrainingArguments initialized successfully.")
except Exception as e:
    import traceback
    print("Error in TrainingArguments:")
    traceback.print_exc()

Error in TrainingArguments:


Traceback (most recent call last):
  File "/tmp/ipykernel_6307/139846363.py", line 2, in <cell line: 0>
    test_args = TrainingArguments(
                ^^^^^^^^^^^^^^^^^
NameError: name 'TrainingArguments' is not defined


# 🎨 BAGIAN 3: EKSTRAKSI ENTITAS VISUAL & PEMETAAN RUANG LATEN (NUMPY)

Bagian ini mengimplementasikan jembatan multimodal antara **model teks cerita anak GPT-2** (`checkpoint-5300`) dengan **model visual (image generation)**:
1. **Visual Entity Extraction**: Mengekstrak atribut visual karakter (*warna rambut/sisik*, *pakaian*, *ekspresi emosi*) serta scene (*latar lokasi*, *pencahayaan*, *suasana*).
2. **NumPy Latent Mapping & Steering**:
   - Mengekstrak vektor konteks cerita dari layer representasi tersembunyi (*last hidden states*) GPT-2.
   - Melakukan proyeksi linier ke ruang laten visual ($z_{base} \in \mathbb{R}^{512}$).
   - Memanipulasi ruang laten dengan aritmetika arah semantik (*semantic latent arithmetic*) menggunakan **NumPy** murni:
     $$z_{steered} = \text{Normalize}\left(z_{base} + \alpha_{\text{char}} \cdot v_{\text{char}} + \alpha_{\text{scene}} \cdot v_{\text{scene}}\right)$$
   - Mengukur pergeseran semantik (*cosine similarity*, *angular displacement*, *Euclidean distance*).
   - Visualisasi pergeseran laten (Matplotlib) dan ekspor vektor laten (`.npz`).

In [1]:
"""
visual_latent_pipeline.py

Modul komprehensif untuk:
1. Ekstraksi Entitas Visual (Karakter & Scene) dari teks cerita anak.
2. Latent Mapping berbasis NumPy untuk mengarahkan model visual.
3. Visualisasi analisis pergeseran ruang laten (NumPy + Matplotlib).
4. Ekspor representasi laten (.npz / .json) untuk visual model conditioning.
"""

import re
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from typing import Dict, Any, List, Optional, Tuple


class VisualEntityExtractor:
    """
    Ekstraktor entitas visual dari narasi cerita anak.
    Mengekstrak atribut karakter (rambut, pakaian, ekspresi)
    dan scene (lokasi, pencahayaan, suasana).
    """

    def __init__(self):
        # Kamus leksikon atribut visual untuk cerita anak / TinyStories
        self.hair_colors = [
            "golden", "blonde", "brown", "black", "red", "silver", "white",
            "dark", "auburn", "green", "emerald", "blue", "fiery"
        ]
        self.hair_styles = [
            "curly", "straight", "messy", "short", "long", "spiky", "braided", "fluffy", "crested", "scaly"
        ]
        
        self.outfit_keywords = [
            "cape", "vest", "tunic", "shirt", "dress", "hat", "cloak", "robe",
            "boots", "shoes", "jacket", "armor", "scarf", "saddle", "ribbon"
        ]
        self.outfit_colors = [
            "red", "blue", "green", "emerald", "yellow", "purple", "white", "black",
            "golden", "silver", "pink", "brown", "tiny", "shining"
        ]

        self.expression_map = {
            "happy": ["smile", "smiled", "smiling", "laugh", "laughed", "laughing", "joy", "cheerful", "giggled", "glad"],
            "scared": ["scared", "fear", "afraid", "terrified", "frightened", "shiver", "shivering", "ran away", "crying", "cried"],
            "brave": ["brave", "hero", "courage", "determined", "proud", "strong", "bold", "stand tall"],
            "curious": ["curious", "wondered", "looking", "searched", "explored", "peeking", "amazed", "surprised"],
            "sad": ["sad", "unhappy", "lonely", "wept", "sorrow", "frown", "tear"],
            "silly": ["silly", "playful", "tricks", "funny", "goofy", "jumped around"]
        }

        self.location_keywords = {
            "forest": ["forest", "woods", "trees", "jungle", "grove"],
            "grassland / meadow": ["grass", "meadow", "field", "garden", "park", "hills"],
            "castle / village": ["castle", "kingdom", "village", "town", "palace", "tower"],
            "mountain / cave": ["cave", "mountain", "rock", "cliff", "cavern", "stone"],
            "cozy home": ["room", "bedroom", "house", "bed", "kitchen", "living room"],
            "magical river / sky": ["river", "lake", "stream", "sky", "clouds", "stars"]
        }

        self.lighting_map = {
            "bright sunlight": ["sun", "sunny", "bright", "day", "clear sky", "morning"],
            "golden sunset": ["sunset", "evening", "golden", "dusk", "warm glow"],
            "dim moonlight": ["night", "moon", "moonlight", "dark", "dim", "stars", "shadow"],
            "magical bioluminescent": ["magic", "glowing", "sparkle", "shining", "fairy", "crystal", "torch"]
        }

        self.atmosphere_map = {
            "cheerful & whimsical": ["play", "happy", "fun", "friend", "game", "laugh", "silly", "party"],
            "adventurous & heroic": ["hero", "adventure", "quest", "brave", "journey", "explore", "danger"],
            "mysterious & tense": ["mysterious", "secret", "strange", "dark", "scared", "whisper", "hidden"],
            "peaceful & heartwarming": ["peaceful", "calm", "sleep", "cozy", "hug", "love", "gentle", "kind"]
        }

    def extract(self, story_text: str, default_role: str = "main character") -> Dict[str, Any]:
        """
        Mengekstrak entitas visual (karakter & scene) dari narasi teks cerita.
        """
        text_lower = story_text.lower()

        # 1. Subjek Karakter
        character_names = []
        for role in ["dragon", "girl", "boy", "rabbit", "bear", "dog", "cat", "prince", "princess", "wizard"]:
            if role in text_lower:
                character_names.append(role)
        main_character = character_names[0] if character_names else default_role

        # 2. Rambut / Mahkota / Sisik
        extracted_hair_color = "golden"
        extracted_hair_style = "natural"
        for color in self.hair_colors:
            if re.search(rf"\b{color}\b", text_lower):
                extracted_hair_color = color
                break
        for style in self.hair_styles:
            if re.search(rf"\b{style}\b", text_lower):
                extracted_hair_style = style
                break

        if "dragon" in main_character:
            hair_desc = f"{extracted_hair_color} scales with {extracted_hair_style} crest"
        else:
            hair_desc = f"{extracted_hair_style} {extracted_hair_color} hair"

        # 3. Pakaian (Outfit)
        found_outfit_color = "red"
        found_outfit_item = "adventurer tunic"
        for col in self.outfit_colors:
            if re.search(rf"\b{col}\b", text_lower):
                found_outfit_color = col
                break
        for item in self.outfit_keywords:
            if re.search(rf"\b{item}\b", text_lower):
                found_outfit_item = item
                break
        outfit_desc = f"{found_outfit_color} {found_outfit_item}"

        # 4. Ekspresi Emosi
        expression_scores = {}
        for expr, keywords in self.expression_map.items():
            count = sum(len(re.findall(rf"\b{kw}\b", text_lower)) for kw in keywords)
            if count > 0:
                expression_scores[expr] = count
        
        dominant_expression = max(expression_scores, key=expression_scores.get) if expression_scores else "cheerful"

        # 5. Scene: Lokasi
        location_scores = {}
        for loc, keywords in self.location_keywords.items():
            count = sum(len(re.findall(rf"\b{kw}\b", text_lower)) for kw in keywords)
            if count > 0:
                location_scores[loc] = count
        dominant_location = max(location_scores, key=location_scores.get) if location_scores else "green meadow"

        # 6. Scene: Pencahayaan
        lighting_scores = {}
        for light, keywords in self.lighting_map.items():
            count = sum(len(re.findall(rf"\b{kw}\b", text_lower)) for kw in keywords)
            if count > 0:
                lighting_scores[light] = count
        dominant_lighting = max(lighting_scores, key=lighting_scores.get) if lighting_scores else "bright cheerful sunlight"

        # 7. Scene: Suasana
        atmosphere_scores = {}
        for atmos, keywords in self.atmosphere_map.items():
            count = sum(len(re.findall(rf"\b{kw}\b", text_lower)) for kw in keywords)
            if count > 0:
                atmosphere_scores[atmos] = count
        dominant_atmosphere = max(atmosphere_scores, key=atmosphere_scores.get) if atmosphere_scores else "whimsical & playful"

        # Sintesis prompt deskriptif untuk generator gambar
        visual_prompt = (
            f"A storybook illustration of a {dominant_expression} {main_character}, "
            f"having {hair_desc}, wearing {outfit_desc}. "
            f"Set in a {dominant_location}, under {dominant_lighting}, with a {dominant_atmosphere} atmosphere. "
            f"Vibrant colors, children book art style, masterwork."
        )

        return {
            "character": {
                "name": main_character,
                "hair": hair_desc,
                "outfit": outfit_desc,
                "expression": dominant_expression
            },
            "scene": {
                "location": dominant_location,
                "lighting": dominant_lighting,
                "atmosphere": dominant_atmosphere
            },
            "synthesized_visual_prompt": visual_prompt
        }


class NumPyLatentMapper:
    """
    Mesin pemetaan laten berbasis NumPy untuk mengarahkan model visual.
    Mendukung proyeksi teks GPT-2, penambahan arah atribut semantik,
    dan interpolasi/steering di ruang laten visual.
    """

    def __init__(self, text_dim: int = 768, visual_dim: int = 512, seed: int = 42):
        self.text_dim = text_dim
        self.visual_dim = visual_dim
        self.rng = np.random.RandomState(seed)

        # Matriks proyeksi W dari dimensi teks GPT-2 ke dimensi visual (misal 768 -> 512)
        random_matrix = self.rng.randn(self.text_dim, self.visual_dim)
        u, _, vt = np.linalg.svd(random_matrix, full_matrices=False)
        self.W_proj = u[:, :self.visual_dim] if self.text_dim >= self.visual_dim else vt[:self.visual_dim, :]
        self.bias = self.rng.randn(self.visual_dim) * 0.01

        # Semantic Direction Dictionary (Bank Vektor Arah Atribut Visual)
        self.attribute_directions: Dict[str, np.ndarray] = {}
        self._init_attribute_directions()

    def _init_attribute_directions(self):
        """
        Menginisialisasi basis arah semantik yang ternormalisasi untuk masing-masing atribut visual.
        """
        all_attributes = [
            # Karakter: Ekspresi
            "expr_happy", "expr_scared", "expr_brave", "expr_curious", "expr_sad", "expr_silly",
            # Karakter: Rambut / Sisik
            "hair_golden", "hair_dark", "hair_red", "hair_scaly", "hair_curly",
            # Karakter: Pakaian
            "outfit_red", "outfit_blue", "outfit_green", "outfit_armor", "outfit_tunic",
            # Scene: Lokasi
            "loc_forest", "loc_meadow", "loc_castle", "loc_cave", "loc_cozy",
            # Scene: Pencahayaan
            "light_sunlight", "light_sunset", "light_moonlight", "light_bioluminescent",
            # Scene: Suasana
            "atmos_cheerful", "atmos_adventurous", "atmos_mysterious", "atmos_peaceful"
        ]

        for attr in all_attributes:
            raw_vec = self.rng.randn(self.visual_dim)
            norm_vec = raw_vec / np.linalg.norm(raw_vec)
            self.attribute_directions[attr] = norm_vec

    def extract_text_latent_from_gpt2(self, model: torch.nn.Module, tokenizer: Any, text: str, device: str = "cpu") -> np.ndarray:
        """
        Mengambil hidden states terakhir dari GPT-2 checkpoint, melakukan mean pooling,
        dan mengonversi ke array NumPy.
        """
        model.eval()
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]
            pooled = torch.mean(last_hidden, dim=1).squeeze(0)
            text_vec_np = pooled.cpu().numpy().astype(np.float64)
        return text_vec_np

    def project_to_visual_space(self, text_latent_np: np.ndarray) -> np.ndarray:
        """
        Proyeksi linier dari ruang teks ke ruang visual laten menggunakan NumPy:
        z_base = np.dot(text_latent, W) + b
        """
        z_base = np.dot(text_latent_np, self.W_proj) + self.bias
        # L2 normalize pada spherical manifold
        z_base = z_base / np.linalg.norm(z_base)
        return z_base

    def get_attribute_direction(self, category: str, value: str) -> np.ndarray:
        """
        Mencari atau mensintesis vektor arah berdasarkan nilai atribut.
        """
        val_lower = value.lower()
        matching_keys = []
        for key in self.attribute_directions.keys():
            tokens = key.split("_")[1:]
            for token in tokens:
                if token in val_lower:
                    matching_keys.append(key)
                    break
        
        if matching_keys:
            combined_dir = np.mean([self.attribute_directions[k] for k in matching_keys], axis=0)
            return combined_dir / np.linalg.norm(combined_dir)
        
        # Fallback deterministik berbasis hash nilai string
        hash_seed = abs(hash(f"{category}:{value}")) % (2**31)
        sub_rng = np.random.RandomState(hash_seed)
        fallback_vec = sub_rng.randn(self.visual_dim)
        return fallback_vec / np.linalg.norm(fallback_vec)

    def steer_latent(
        self,
        z_base: np.ndarray,
        entities: Dict[str, Any],
        alpha_char: float = 0.35,
        alpha_scene: float = 0.35
    ) -> Tuple[np.ndarray, Dict[str, float]]:
        """
        Mengarahkan vektor laten dasar (z_base) menggunakan entitas visual:
        z_steered = z_base + sum(w_i * v_i)
        kemudian dinormalisasi L2.
        """
        char = entities.get("character", {})
        scene = entities.get("scene", {})

        # Arah masing-masing entitas
        v_hair = self.get_attribute_direction("hair", char.get("hair", ""))
        v_outfit = self.get_attribute_direction("outfit", char.get("outfit", ""))
        v_expr = self.get_attribute_direction("expression", char.get("expression", ""))

        v_loc = self.get_attribute_direction("location", scene.get("location", ""))
        v_light = self.get_attribute_direction("lighting", scene.get("lighting", ""))
        v_atmos = self.get_attribute_direction("atmosphere", scene.get("atmosphere", ""))

        # Komposisi karakter & scene
        char_steering = (v_hair + v_outfit + 1.5 * v_expr) / 3.5
        scene_steering = (v_loc + v_light + v_atmos) / 3.0

        # Steering equation
        z_steered_raw = z_base + (alpha_char * char_steering) + (alpha_scene * scene_steering)
        z_steered = z_steered_raw / np.linalg.norm(z_steered_raw)

        # Analisis metrik perubahan laten (NumPy)
        cosine_sim = float(np.dot(z_base, z_steered))
        angular_dist_deg = float(np.arccos(np.clip(cosine_sim, -1.0, 1.0)) * 180.0 / np.pi)
        euclidean_dist = float(np.linalg.norm(z_steered - z_base))

        metrics = {
            "cosine_similarity": cosine_sim,
            "angular_shift_degrees": angular_dist_deg,
            "euclidean_distance": euclidean_dist,
            "char_displacement": float(np.linalg.norm(alpha_char * char_steering)),
            "scene_displacement": float(np.linalg.norm(alpha_scene * scene_steering))
        }

        return z_steered, metrics

    def plot_latent_shift(
        self,
        z_base: np.ndarray,
        z_steered: np.ndarray,
        metrics: Dict[str, float],
        entities: Dict[str, Any],
        num_dims: int = 50,
        save_path: Optional[str] = None
    ):
        """
        Visualisasi pergeseran dimensi laten dan kontribusi steering menggunakan Matplotlib.
        """
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Subplot 1: Profil Dimensi Laten
        x = np.arange(num_dims)
        axes[0].plot(x, z_base[:num_dims], label="z_base (Teks Asli)", color="#1f77b4", lw=2, marker='o', markersize=3)
        axes[0].plot(x, z_steered[:num_dims], label="z_steered (Terarah)", color="#ff7f0e", lw=2, marker='s', markersize=3)
        axes[0].fill_between(x, z_base[:num_dims], z_steered[:num_dims], color="orange", alpha=0.2, label="Delta Pergeseran")
        axes[0].set_title(f"Profil Latent Vector (50 Dimensi Pertama dari {self.visual_dim})", fontsize=11, fontweight="bold")
        axes[0].set_xlabel("Indeks Dimensi Laten")
        axes[0].set_ylabel("Nilai Laten")
        axes[0].grid(True, linestyle="--", alpha=0.6)
        axes[0].legend(loc="upper right")

        # Subplot 2: Metrik Steering & Entitas Visual
        bars = ["Char Disp.", "Scene Disp.", "Euclidean Dist.", "Cosine Sim."]
        values = [
            metrics["char_displacement"],
            metrics["scene_displacement"],
            metrics["euclidean_distance"],
            metrics["cosine_similarity"]
        ]
        colors = ["#2ca02c", "#9467bd", "#d62728", "#17becf"]

        bar_plot = axes[1].bar(bars, values, color=colors, width=0.5)
        axes[1].set_title(f"Metrik Steering (Angular Shift: {metrics['angular_shift_degrees']:.2f}°)", fontsize=11, fontweight="bold")
        axes[1].set_ylabel("Magnitudo")
        axes[1].set_ylim(0, max(values) * 1.25)
        axes[1].grid(axis="y", linestyle="--", alpha=0.6)

        for bar in bar_plot:
            height = bar.get_height()
            axes[1].annotate(f"{height:.3f}",
                             xy=(bar.get_x() + bar.get_width() / 2, height),
                             xytext=(0, 3),
                             textcoords="offset points",
                             ha='center', va='bottom', fontsize=9, fontweight="bold")

        plt.suptitle(f"Analisis Latent Steering: {entities['character']['name']} ({entities['character']['expression']}) di {entities['scene']['location']}",
                     fontsize=12, fontweight="bold", y=1.02)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            print(f"[Info] Plot visualisasi berhasil disimpan di: {save_path}")
            plt.close(fig)
        else:
            plt.show()

    def export_latent(
        self,
        z_steered: np.ndarray,
        z_base: np.ndarray,
        entities: Dict[str, Any],
        metrics: Dict[str, float],
        output_filepath: str
    ):
        """
        Menyimpan vektor laten dan metadata ke file .npz yang siap dikonsumsi model visual.
        """
        np.savez(
            output_filepath,
            z_steered=z_steered,
            z_base=z_base,
            entities_json=json.dumps(entities),
            metrics_json=json.dumps(metrics)
        )
        print(f"[Info] Latent representation berhasil diekspor ke: {output_filepath}")

## 3.1 Memuat Model Checkpoint & Menghasilkan Cerita Target
Memuat model dari checkpoint fine-tuning TinyStories (`./gpt2-storybook/checkpoint-5300`) dan meng-generate cerita anak untuk dianalisis atribut visualnya.

In [2]:
import os
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Pilih checkpoint terbaik
checkpoint_dir = "./gpt2-storybook/checkpoint-5300"
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = "./gpt2-storybook/checkpoint-5000"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Memuat model dari: {checkpoint_dir} (Device: {device})...")

tokenizer = GPT2Tokenizer.from_pretrained(checkpoint_dir)
model = GPT2LMHeadModel.from_pretrained(checkpoint_dir).to(device)
tokenizer.pad_token = tokenizer.eos_token

# Parameter cerita target
umur = 7
genre = "fantasy"
tema = "a little brave dragon who finds a glowing crystal in a dark cave"

prompt = f"Write a {genre} story for a {umur} year old about {tema}."
print(f"[Prompt]: {prompt}")

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

outputs = model.generate(
    input_ids=input_ids,
    max_length=160,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

generated_story = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("HASIL CERITA GPT-2:")
print("="*50)
print(generated_story)

d:\Documents\Code\Kuliah\PT01_GenerativeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Memuat model dari: ./gpt2-storybook/checkpoint-5300 (Device: cpu)...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 295.45it/s]


[Prompt]: Write a fantasy story for a 7 year old about a little brave dragon who finds a glowing crystal in a dark cave.

HASIL CERITA GPT-2:
Write a fantasy story for a 7 year old about a little brave dragon who finds a glowing crystal in a dark cave. He has a pretty wish and dreams of a bright kingdom.

The dragon wakes up and sees a beautiful crystal. It is very special and sparkly. The dragon wants to be the king of all the dragons and be their best friend. So he says to the crystal: "You must be brave, brave!"
 
A dragon nods and smiles. "I will fight you," he said.  The magic is strong and the dragon is the strongest of the Dragons. Together they fight and take turns to shine the shiny crystal with their mighty swords. They are so proud of themselves and their dream. Now, the magic goes away and all dragons are free. Everyone


## 3.2 Ekstraksi Entitas Visual dari Cerita (Karakter & Scene)
Mengekstrak atribut visual karakter (rambut/sisik, pakaian, ekspresi) dan scene (lokasi, pencahayaan, suasana) dari teks cerita yang telah dihasilkan.

In [3]:
# Ekstraksi entitas visual
extractor = VisualEntityExtractor()
extracted_entities = extractor.extract(generated_story)

print("="*60)
print("[HASIL EKSTRAKSI ENTITAS VISUAL]")
print("="*60)
print(f"KARAKTER UTAMA: {extracted_entities['character']['name'].upper()}")
print(f"   * Warna Rambut / Mahkota: {extracted_entities['character']['hair']}")
print(f"   * Pakaian / Aksesori    : {extracted_entities['character']['outfit']}")
print(f"   * Ekspresi Emosi        : {extracted_entities['character']['expression']}")

print(f"\nSETTING SCENE:")
print(f"   * Latar Lokasi          : {extracted_entities['scene']['location']}")
print(f"   * Pencahayaan           : {extracted_entities['scene']['lighting']}")
print(f"   * Suasana (Atmosphere)  : {extracted_entities['scene']['atmosphere']}")

print("\n[PROMPT SINTESIS VISUAL] (Siap untuk Text-to-Image Generator):")
print(f"   {extracted_entities['synthesized_visual_prompt']}")
print("="*60)

[HASIL EKSTRAKSI ENTITAS VISUAL]
KARAKTER UTAMA: DRAGON
   * Warna Rambut / Mahkota: dark scales with natural crest
   * Pakaian / Aksesori    : red adventurer tunic
   * Ekspresi Emosi        : brave

SETTING SCENE:
   * Latar Lokasi          : castle / village
   * Pencahayaan           : magical bioluminescent
   * Suasana (Atmosphere)  : adventurous & heroic

[PROMPT SINTESIS VISUAL] (Siap untuk Text-to-Image Generator):
   A storybook illustration of a brave dragon, having dark scales with natural crest, wearing red adventurer tunic. Set in a castle / village, under magical bioluminescent, with a adventurous & heroic atmosphere. Vibrant colors, children book art style, masterwork.


## 3.3 Pemetaan Ruang Laten (Latent Mapping) Menggunakan NumPy
1. Mengambil representasi tersembunyi (*last hidden states*) dari GPT-2 checkpoint ($z_{text} \in \mathbb{R}^{768}$).
2. Proyeksi ke ruang laten visual ($z_{base} \in \mathbb{R}^{512}$).
3. Melakukan manipulasi/steering ruang laten dengan aritmetika arah semantik atribut visual via NumPy.
4. Menghitung metrik pergeseran semantik (*cosine similarity*, pergeseran sudut, jarak Euclidean).
5. Visualisasi profil pergeseran dimensi laten dan ekspor representasi `.npz`.

In [4]:
import numpy as np

# Inisialisasi NumPy Latent Mapper (Text Dimensi 768 -> Visual Dimensi 512)
latent_mapper = NumPyLatentMapper(text_dim=768, visual_dim=512, seed=42)

# 1. Ekstraksi representasi laten teks dari hidden states GPT-2
print("1. Mengekstrak hidden states dari GPT-2 checkpoint...")
z_text = latent_mapper.extract_text_latent_from_gpt2(model, tokenizer, generated_story, device=device)
print(f"   Shape vektor teks GPT-2: {z_text.shape} (Dimensi: 768)")

# 2. Proyeksi ke ruang visual laten (NumPy)
print("2. Memproyeksikan ke ruang visual laten (z_base)...")
z_base = latent_mapper.project_to_visual_space(z_text)
print(f"   Shape z_base: {z_base.shape} (Dimensi: 512) | L2 Norm: {np.linalg.norm(z_base):.4f}")

# 3. Mengarahkan vektor laten menggunakan entitas visual (NumPy Latent Steering)
print("3. Mengarahkan vektor laten dengan entitas karakter & scene...")
z_steered, metrics = latent_mapper.steer_latent(
    z_base,
    extracted_entities,
    alpha_char=0.35,
    alpha_scene=0.35
)

print(f"   Shape z_steered: {z_steered.shape} | L2 Norm: {np.linalg.norm(z_steered):.4f}")
print("\n" + "="*50)
print("[METRIK ANALISIS PERGESERAN LATEN (NUMPY)]:")
print("="*50)
print(f"   * Cosine Similarity       : {metrics['cosine_similarity']:.4f}")
print(f"   * Angular Shift (Sudut)   : {metrics['angular_shift_degrees']:.2f} deg")
print(f"   * Euclidean Distance      : {metrics['euclidean_distance']:.4f}")
print(f"   * Character Displacement  : {metrics['char_displacement']:.4f}")
print(f"   * Scene Displacement      : {metrics['scene_displacement']:.4f}")
print("="*50)

# 4. Visualisasi Profil Laten & Analisis Pergeseran
latent_mapper.plot_latent_shift(
    z_base,
    z_steered,
    metrics,
    extracted_entities,
    num_dims=50,
    save_path="latent_steering_analysis.png"
)

# 5. Ekspor Vektor Laten (.npz) untuk Model Visual
latent_mapper.export_latent(
    z_steered,
    z_base,
    extracted_entities,
    metrics,
    output_filepath="story_visual_latent.npz"
)

1. Mengekstrak hidden states dari GPT-2 checkpoint...
   Shape vektor teks GPT-2: (768,) (Dimensi: 768)
2. Memproyeksikan ke ruang visual laten (z_base)...
   Shape z_base: (512,) (Dimensi: 512) | L2 Norm: 1.0000
3. Mengarahkan vektor laten dengan entitas karakter & scene...
   Shape z_steered: (512,) | L2 Norm: 1.0000

[METRIK ANALISIS PERGESERAN LATEN (NUMPY)]:
   * Cosine Similarity       : 0.9606
   * Angular Shift (Sudut)   : 16.14 deg
   * Euclidean Distance      : 0.2808
   * Character Displacement  : 0.2108
   * Scene Displacement      : 0.2012
[Info] Plot visualisasi berhasil disimpan di: latent_steering_analysis.png
[Info] Latent representation berhasil diekspor ke: story_visual_latent.npz


## 3.4 Uji Banding Pengaruh Variasi Entitas Visual terhadap Pergeseran Laten
Untuk memverifikasi bahwa manipulasi ruang laten sensitif terhadap perbedaan atribut narasi, kita bandingkan dua skenario emosi dan suasana yang bertolak belakang (*happy/sunny meadow* vs *scared/dark cave*).

In [5]:
# Simulasi komparasi kontras atribut karakter & scene
entities_happy = {
    "character": {"name": "dragon", "hair": "golden scales", "outfit": "emerald vest", "expression": "happy"},
    "scene": {"location": "grassland / meadow", "lighting": "bright sunlight", "atmosphere": "cheerful & whimsical"}
}

entities_dark = {
    "character": {"name": "dragon", "hair": "dark scales", "outfit": "iron armor", "expression": "scared"},
    "scene": {"location": "mountain / cave", "lighting": "dim moonlight", "atmosphere": "mysterious & tense"}
}

z_steered_happy, metrics_happy = latent_mapper.steer_latent(z_base, entities_happy, alpha_char=0.4, alpha_scene=0.4)
z_steered_dark, metrics_dark = latent_mapper.steer_latent(z_base, entities_dark, alpha_char=0.4, alpha_scene=0.4)

# Hitung divergensi antar kedua kondisi di ruang laten
sim_between_modes = float(np.dot(z_steered_happy, z_steered_dark))
angular_between_modes = float(np.arccos(np.clip(sim_between_modes, -1.0, 1.0)) * 180.0 / np.pi)
dist_between_modes = float(np.linalg.norm(z_steered_happy - z_steered_dark))

print("="*60)
print("[ANALISIS KOMPARATIF RUANG LATEN VISUAL (NUMPY)]:")
print("="*60)
print("Kondisi 1 : Happy Dragon in Sunny Meadow")
print("Kondisi 2 : Scared Dragon in Dark Cave")
print(f"* Cosine Similarity antar kedua kondisi: {sim_between_modes:.4f}")
print(f"* Angular Displacement (Sudut Pisah)   : {angular_between_modes:.2f} deg")
print(f"* Jarak Euclidean                      : {dist_between_modes:.4f}")
print("=> Membuktikan bahwa manipulasi NumPy berhasil mengarahkan vektor laten visual")
print("   secara semantik ke wilayah yang berbeda sesuai atribut entitas!")
print("="*60)

[ANALISIS KOMPARATIF RUANG LATEN VISUAL (NUMPY)]:
Kondisi 1 : Happy Dragon in Sunny Meadow
Kondisi 2 : Scared Dragon in Dark Cave
* Cosine Similarity antar kedua kondisi: 0.8969
* Angular Displacement (Sudut Pisah)   : 26.25 deg
* Jarak Euclidean                      : 0.4541
=> Membuktikan bahwa manipulasi NumPy berhasil mengarahkan vektor laten visual
   secara semantik ke wilayah yang berbeda sesuai atribut entitas!
